In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
print(os.listdir('/kaggle/input'))

['datasets']


In [3]:
import os
print(os.listdir('/kaggle/input/datasets'))

['beautifulminnd']


In [4]:
import os
print(os.listdir('/kaggle/input/datasets/beautifulminnd'))

['tau-urban-acoustic-scenes-2022-mobile-development']


In [5]:
import os
base = '/kaggle/input/datasets/beautifulminnd/tau-urban-acoustic-scenes-2022-mobile-development'
print(os.listdir(base))

['TAU-urban-acoustic-scenes-2022-mobile-development.audio.8', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.15', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.1', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.9', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.10', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.5', 'TAU-urban-acoustic-scenes-2022-mobile-development.meta', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.16', 'TAU-urban-acoustic-scenes-2022-mobile-development.doc', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.12', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.2', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.4', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.6', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.3', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.7', 'TAU-urban-acoustic-scenes-2022-mobile-development.audio.13', 'TAU-urban-acoustic-scene

In [6]:
import glob

base = '/kaggle/input/datasets/beautifulminnd/tau-urban-acoustic-scenes-2022-mobile-development'
wavs = glob.glob(f"{base}/**/audio/*.wav", recursive=True)
print(f"Toplam wav: {len(wavs)}")
print(f"Örnek yol: {wavs[0]}")

Toplam wav: 230350
Örnek yol: /kaggle/input/datasets/beautifulminnd/tau-urban-acoustic-scenes-2022-mobile-development/TAU-urban-acoustic-scenes-2022-mobile-development.audio.8/TAU-urban-acoustic-scenes-2022-mobile-development/audio/park-vienna-104-2962-5-a.wav


In [1]:
import os, glob
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Cihaz: {device}")

✅ Cihaz: cuda


In [5]:
BASE = '/kaggle/input/datasets/beautifulminnd/tau-urban-acoustic-scenes-2022-mobile-development'
META_CSV = f"{BASE}/TAU-urban-acoustic-scenes-2022-mobile-development.meta/TAU-urban-acoustic-scenes-2022-mobile-development/meta.csv"

# Tüm wav dosyalarını tara
all_wavs = glob.glob(f"{BASE}/**/audio/*.wav", recursive=True)
wav_dict = {os.path.basename(p): p for p in all_wavs}
print(f"✅ Toplam wav: {len(wav_dict)}")

# Meta CSV'yi oku
meta = pd.read_csv(META_CSV, sep='\t')
meta['fname'] = meta['filename'].apply(os.path.basename)
meta['city']  = meta['fname'].apply(lambda x: x.split('-')[1])
print(f"✅ Meta kayıt: {len(meta)}")

# Şehre göre train/val böl
VAL_CITIES   = {'barcelona', 'helsinki'}
TRAIN_CITIES = set(meta['city'].unique()) - VAL_CITIES

train_list, val_list = [], []
for _, row in meta.iterrows():
    fname = row['fname']
    if fname not in wav_dict:
        continue
    city  = row['city']
    label = row['scene_label']
    if city in VAL_CITIES:
        val_list.append((wav_dict[fname], label))
    elif city in TRAIN_CITIES:
        train_list.append((wav_dict[fname], label))

unique_labels = sorted(set(l for _, l in train_list + val_list))
label_to_idx  = {label: i for i, label in enumerate(unique_labels)}

print(f"\n🚂 Train: {len(train_list)} | 🔍 Val: {len(val_list)}")
print(f"📋 Sınıflar: {unique_labels}")


def process_audio(audio_path, augment=False):
    waveform, sample_rate = torchaudio.load(audio_path)

    if augment:
        # Hafif zaman kaydırma
        shift = int(sample_rate * 0.05)
        waveform = torch.roll(waveform, shifts=shift, dims=1)
        # Hafif ses seviyesi değiştirme
        waveform = waveform * (0.9 + 0.2 * torch.rand(1).item())

    mel_db = T.AmplitudeToDB()(
        T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=2048,
            hop_length=512,
            n_mels=64
        )(waveform)
    )

    # Hafif SpecAugment
    if augment:
        mel_db = T.FrequencyMasking(freq_mask_param=5)(mel_db)
        mel_db = T.TimeMasking(time_mask_param=10)(mel_db)

    input_tensor = nn.functional.interpolate(
        mel_db.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False
    ).squeeze(0)

    if input_tensor.shape[0] == 1:
        input_tensor = input_tensor.repeat(3, 1, 1)

    return input_tensor


class TAUDataset(Dataset):
    def __init__(self, data_list, label_to_idx, augment=False):
        self.data_list     = data_list
        self.label_to_idx  = label_to_idx
        self.augment       = augment
        self.unique_labels = list(label_to_idx.keys())

    def __len__(self): return len(self.data_list)

    def __getitem__(self, idx):
        path, label = self.data_list[idx]
        return process_audio(path, self.augment), torch.tensor(self.label_to_idx[label], dtype=torch.long)


train_dataset = TAUDataset(train_list, label_to_idx, augment=True)
val_dataset   = TAUDataset(val_list,   label_to_idx, augment=False)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print("✅ DataLoader hazır!")

✅ Toplam wav: 230350
✅ Meta kayıt: 230350

🚂 Train: 182510 | 🔍 Val: 47840
📋 Sınıflar: ['airport', 'bus', 'metro', 'metro_station', 'park', 'public_square', 'shopping_mall', 'street_pedestrian', 'street_traffic', 'tram']
✅ DataLoader hazır!


In [9]:
class MobileNetTransformer(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        self.feature_extractor = mobilenet.features
        self.hidden_dim = 1280
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_dim, nhead=8, batch_first=True, dropout=0.3
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        f = self.feature_extractor(x)
        f = f.mean(dim=2).permute(0, 2, 1)
        return self.classifier(self.transformer(f).mean(dim=1))


model     = MobileNetTransformer(num_classes=len(unique_labels)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

print(f"✅ Model hazır | Parametre: {sum(p.numel() for p in model.parameters()):,}")
print(f"🖥️  Cihaz: {device}\n")

best_val_acc = 0

for epoch in range(10):
    model.train()
    correct, total = 0, 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        correct += (torch.max(outputs,1)[1] == labels).sum().item()
        total   += labels.size(0)
        if (i+1) % 200 == 0:
            print(f"  Epoch {epoch+1} | Adım {i+1}/{len(train_loader)} | Kayıp: {loss.item():.4f}")

    scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_correct += (torch.max(model(images),1)[1] == labels).sum().item()
            val_total   += labels.size(0)

    train_acc = 100 * correct / total
    val_acc   = 100 * val_correct / val_total

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print(f"  💾 En iyi model kaydedildi!")

    print(f"\n{'='*55}")
    print(f"Epoch {epoch+1:2d}/10 | Train: %{train_acc:.1f} | Val: %{val_acc:.1f} | En iyi: %{best_val_acc:.1f}")
    print(f"{'='*55}\n")

print("🎉 Eğitim tamamlandı!")
print(f"🏆 En iyi val doğruluğu: %{best_val_acc:.1f}")

✅ Model hazır | Parametre: 26,174,986
🖥️  Cihaz: cuda



KeyboardInterrupt: 